In [ ]:
# @title 1. Clone LTX-Video repo + install (native inference path)
import sys, os
if not os.path.exists('/content/ltx'):
    !git clone --depth 1 https://github.com/Lightricks/LTX-Video.git /content/ltx
sys.path.insert(0, '/content/ltx')
!pip install -q -e /content/ltx
!pip install -q imageio imageio-ffmpeg av
try:
    from ltx_video.inference import InferenceConfig
    print('OK: ltx_video importable')
except Exception as e:
    print('IMPORT FAILED:', repr(e))

In [ ]:
# @title 2. Upload vault_door.jpeg
# Drag vault_door.jpeg into the LEFT sidebar file browser, then run this.
import os
IMAGE_PATH = '/content/vault_door.jpeg'
assert os.path.exists(IMAGE_PATH), 'Upload vault_door.jpeg to the left sidebar first'
print('image present:', IMAGE_PATH)

In [ ]:
# @title 3. Run LTX 2B-Distilled image-to-video
# VERIFIED field names from ltx_video/inference.py InferenceConfig:
#   prompt, negative_prompt, conditioning_media_paths (List[str]),
#   conditioning_start_frames (List[int], REQUIRED with conditioning_media_paths),
#   height, width, num_frames, seed, pipeline_config, output_path (a FOLDER)
# Distilled 2B: 8 steps, guidance 1, no STG. Free, fits Colab T4 (15GB).
import sys
if '/content/ltx' not in sys.path:
    sys.path.insert(0, '/content/ltx')
from ltx_video.inference import infer, InferenceConfig

PROMPT = (
    'Paper-collage animation style, flat illustrated vault door made of cut paper layers, '
    'the paper pieces shift and lift slightly with a gentle hand-made stop-motion drift, '
    'soft paper shadows, warm editorial color palette, Vox-style explainer motion, '
    'subtle slow camera push-in, no 3D, no photorealism'
)
NEG = 'photorealistic, 3d render, CGI, jitter, flicker, distorted, blurry'

OUT_DIR = '/content/ltx_out'
cfg = InferenceConfig(
    prompt=PROMPT,
    negative_prompt=NEG,
    conditioning_media_paths=[IMAGE_PATH],
    conditioning_start_frames=[0],
    height=1280, width=704,
    num_frames=121,
    seed=42,
    pipeline_config='configs/ltxv-2b-0.9.6-distilled.yaml',
    output_path=OUT_DIR,
)
infer(config=cfg)
print('inference done ->', OUT_DIR)

In [ ]:
# @title 4. Download result (auto-named video_output_*.mp4 inside OUT_DIR)
import os, glob
from google.colab import files
OUT_DIR = '/content/ltx_out'
vids = sorted(glob.glob(os.path.join(OUT_DIR, 'video_output_*.mp4')))
if vids:
    src = vids[0]
    !cp "{src}" /content/ltx_vault.mp4
    files.download('/content/ltx_vault.mp4')
    print('Download triggered:', src)
    print('Move to ~/Downloads/vox/ltx_vault.mp4 and ask Hermes to verify >=60%.')
else:
    print('No video_output_*.mp4 found in', OUT_DIR, '-> check cell 3 output for errors.')